 #**Hey everyone how is it going** 

In [1]:
-- TABLE 1 - ENERGY SOURCES
CREATE TABLE energy_sources_clean AS 
SELECT * 
FROM '01_energy_sources_raw.csv';


UPDATE energy_sources_clean
	SET energy_source = UPPER(TRIM(energy_source)),
	
	standard_unit = CASE WHEN standard_unit ILIKE '%mwh%' THEN 'MWh'
	                     WHEN standard_unit ILIKE '%litres%' THEN 'L'
	                	 ELSE standard_unit END;

ALTER TABLE energy_sources_clean
	ADD COLUMN MJ_per_unit NUMERIC;

UPDATE energy_sources_clean
   SET MJ_per_unit = CASE WHEN standard_unit = 'MWh' THEN 3600
	                      WHEN standard_unit = 'L'   THEN 40
	                      WHEN standard_unit = 'GJ'  THEN 1000
	                      ELSE MJ_per_unit END;

ALTER TABLE energy_sources_clean
	ADD COLUMN emissions_factor_tco2e_per_MJ DOUBLE;

UPDATE energy_sources_clean
   SET emissions_factor_tco2e_per_MJ = emissions_factor_tco2e_per_unit/MJ_per_unit;
	                                    

-- TABLE 2 - OPERATION HIERARCHY
CREATE TABLE operation_hierarchy_clean AS 
SELECT * 
FROM '02_operation_hierarchy_raw.csv';

UPDATE operation_hierarchy_clean 
	SET main_operation = TRIM(UPPER(main_operation)),
	    sub_operation = TRIM(sub_operation);

UPDATE operation_hierarchy_clean
	SET energy_criticality = 'High'
	WHERE TRIM(energy_criticality) IS NULL;


-- TABLE 3 - DAILY PRODUCTION 
CREATE TABLE daily_production_clean AS 
SELECT *
FROM '03_daily_production_raw.csv';


ALTER TABLE daily_production_clean
	ALTER COLUMN production_date TYPE DATE
	USING TRY_STRPTIME(
    TRIM(production_date),
    ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d','%m-%d-%Y','%m/%d/%Y','%d-%B-%Y','%d-%b-%Y']
	)::DATE;


ALTER TABLE daily_production_clean
	ALTER COLUMN metric_value TYPE NUMERIC
	USING CAST(
	      NULLIF(REPLACE(TRIM(metric_value),',',''),'')
		  AS NUMERIC);

UPDATE daily_production_clean
		SET metric_name = TRIM(UPPER(metric_name)),
	        
	        unit = CASE WHEN unit ILIKE '%tonnes%' THEN 't'
	                    WHEN unit ILIKE '%c%' THEN '%'
	                    WHEN unit ILIKE '%p%' THEN '%'
	                    ELSE unit END; 



UPDATE daily_production_clean
	SET metric_value = CASE WHEN metric_value < 0 THEN ABS(metric_value)
	                   WHEN metric_name = 'COPPER CONCENTRATE PRODUCED' AND metric_value > 1000 THEN metric_value/10
	                   WHEN metric_name = 'COPPER RECOVERY' AND metric_value > 100 THEN metric_value/10
	                   WHEN metric_name = 'ORE PROCESSED' AND metric_value > 40000 THEN metric_value/10
	                   ELSE metric_value END;


-- TABLE 4 - DAILY ENERGY CONSUMPTION
CREATE TABLE daily_energy_consumption_clean AS 
SELECT *
FROM '04_daily_energy_consumption_raw.csv';

ALTER TABLE daily_energy_consumption_clean
	ALTER COLUMN consumption_date TYPE DATE
	USING TRY_STRPTIME(
    TRIM(consumption_date),
    ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d','%m-%d-%Y','%m/%d/%Y','%d-%B-%Y','%d-%b-%Y']
	)::DATE;


UPDATE daily_energy_consumption_clean 
	SET consumption_quantity = CASE WHEN TRIM(consumption_quantity) LIKE '' THEN NULL 
	                                WHEN REGEXP_MATCHES(TRIM(consumption_quantity),'[^0-9.,]') THEN NULL
	                                ELSE REPLACE(TRIM(consumption_quantity),',','')
	                                END,
	
	    reported_unit = CASE WHEN reported_unit ILIKE '%mwh%' THEN 'MWh'
	                         WHEN reported_unit ILIKE '%l%' THEN 'L'
	                         ELSE TRIM(UPPER(reported_unit)) END;
	                	
	
ALTER TABLE daily_energy_consumption_clean 
	ALTER COLUMN consumption_quantity TYPE NUMERIC
	USING CAST (consumption_quantity AS NUMERIC);

UPDATE daily_energy_consumption_clean 
	SET consumption_quantity = ABS(consumption_quantity)
	WHERE consumption_quantity < 0;

ALTER TABLE daily_energy_consumption_clean 
	ADD COLUMN energy_source VARCHAR;

ALTER TABLE daily_energy_consumption_clean 
	ADD COLUMN consumption_quantity_MJ NUMERIC;

UPDATE daily_energy_consumption_clean AS d
	SET energy_source = e.energy_source,
	    consumption_quantity_MJ = consumption_quantity*e.MJ_per_unit
	                              FROM energy_sources_clean AS e
	                              WHERE e.energy_source_id = d.energy_source_id;


UPDATE daily_energy_consumption_clean AS d
	SET consumption_quantity = 
	CASE WHEN sub_operation = 'Blast-hole drilling' AND consumption_quantity > 15000 THEN consumption_quantity/10
	     WHEN sub_operation = 'Ore hauling' AND consumption_quantity > 50000 THEN consumption_quantity/10
	     WHEN sub_operation = 'Waste hauling' AND consumption_quantity > 70000 THEN consumption_quantity/10
	     WHEN sub_operation = 'Secondary crushing' AND consumption_quantity > 150 THEN consumption_quantity/10
	     WHEN sub_operation = 'Tailings pumping' AND consumption_quantity > 200 THEN consumption_quantity/10
	     WHEN sub_operation = 'Reclaim-water pumping' AND consumption_quantity > 100 THEN consumption_quantity/10
	     WHEN sub_operation = 'Administration and camp' AND consumption_quantity > 60 THEN consumption_quantity/10
	     WHEN sub_operation = 'Pit dewatering' AND consumption_quantity > 50 THEN consumption_quantity/10
	     WHEN sub_operation = 'Haul-road maintenance' AND consumption_quantity > 9000 THEN consumption_quantity/10
	     WHEN sub_operation = 'Stockpile Reclaim' AND consumption_quantity > 4000 THEN consumption_quantity/10
	     WHEN sub_operation = 'Ball milling' AND consumption_quantity > 500 THEN consumption_quantity/10
	     WHEN sub_operation = 'Concentrate thickening' AND consumption_quantity > 100 THEN consumption_quantity/10
	     WHEN sub_operation = 'Concentrate filtration' AND consumption_quantity > 100 THEN consumption_quantity/10
	     WHEN sub_operation = 'Concentrate drying' AND consumption_quantity > 200 THEN consumption_quantity/10
	     WHEN sub_operation = 'Tailings thickening' AND consumption_quantity > 100 THEN consumption_quantity/10
	     WHEN sub_operation = 'Workshop maintenance' AND consumption_quantity > 3000 THEN consumption_quantity/10
	     WHEN sub_operation = 'Lighting and communications' AND consumption_quantity > 30 THEN consumption_quantity/10
	     WHEN sub_operation = 'Laboratory' AND consumption_quantity > 20 THEN consumption_quantity/10
	     WHEN sub_operation = 'Dozing and grading' AND consumption_quantity > 20000 THEN consumption_quantity/10
	     ELSE consumption_quantity END
	FROM operation_hierarchy_clean AS o 
	WHERE d.operation_id = o.operation_id;


-- TABLE 5 - MONTHLY ENERGY CONSUMPTION
CREATE TABLE monthly_energy_costs_clean AS 
SELECT *
FROM '05_monthly_energy_costs_raw.csv';
 
ALTER TABLE monthly_energy_costs_clean
ALTER COLUMN billing_month TYPE DATE
USING TRY_STRPTIME(
    TRIM(billing_month),
    ['%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y', '%Y/%m/%d','%m-%d-%Y','%m/%d/%Y','%m/%Y','%m-%Y','%Y/%m','%Y-%m','%B-%Y','%b-%Y']
)::DATE; 

UPDATE monthly_energy_costs_clean 
	SET unit_cost_aud = TRIM(REPLACE(unit_cost_aud,'AUD',''))
	WHERE unit_cost_aud LIKE '%AUD%';

ALTER TABLE monthly_energy_costs_clean
ALTER COLUMN unit_cost_aud TYPE NUMERIC
USING CAST (unit_cost_aud AS NUMERIC);

UPDATE monthly_energy_costs_clean AS m
	SET unit_cost_aud = (SELECT AVG(m2.unit_cost_aud)
	                    FROM monthly_energy_costs_clean AS m2
	                    WHERE m2.energy_source_id = m.energy_source_id)
	                    WHERE unit_cost_aud IS NULL; 

UPDATE monthly_energy_costs_clean 
	SET billed_unit = CASE WHEN billed_unit ILIKE '%mwh%' THEN 'MWh'
	                       ELSE TRIM(UPPER(billed_unit)) END;

ALTER TABLE monthly_energy_costs_clean
ADD COLUMN cost_per_MJ DOUBLE; 
	        
UPDATE monthly_energy_costs_clean AS m
	SET cost_per_MJ =  unit_cost_aud/e.MJ_per_unit,
	 
	    supplier = CASE WHEN supplier ILIKE '%Sun%' THEN 'SUNWEST PPA'
		                ELSE TRIM(UPPER(supplier)) END
					   
	                FROM energy_sources_clean AS e
	                WHERE e.energy_source_id = m.energy_source_id;



--DATA ANALYSIS
WITH cost AS 
	(SELECT  
	     d.operation_id,
	     d.energy_source_id,
	     d.consumption_quantity_MJ/1000 AS consumption_quantity_MJ,
	     m.cost_per_MJ,
	 	((d.consumption_quantity_MJ/1000) * m.cost_per_MJ) AS energy_cost
	 FROM monthly_energy_costs_clean AS m
	 INNER JOIN daily_energy_consumption_clean AS d
	 ON d.energy_source_id = m.energy_source_id
	 WHERE EXTRACT(MONTH FROM d.consumption_date) = EXTRACT(MONTH FROM m.billing_month)),


	
co2emissions_energyconsumption_energycost AS 
	(SELECT 
	
	   o.main_operation, 
	
	   o.sub_operation, 

	   e.energy_source,
	
	   SUM(d.consumption_quantity_MJ/1000) AS total_consumption_MJ,
	
	   SUM(e.emissions_factor_tco2e_per_MJ * (d.consumption_quantity_MJ/1000)) AS total_tco2e_emissions,
	
	   SUM(c.energy_cost) AS total_energy_cost
	
	FROM operation_hierarchy_clean AS o
	
	INNER JOIN daily_energy_consumption_clean AS d
	ON o.operation_id = d.operation_id
	
	INNER JOIN energy_sources_clean AS e
	ON e.energy_source_id = d.energy_source_id
	
	INNER JOIN cost AS c
	ON c.operation_id = o.operation_id
	AND c.energy_source_id = d.energy_source_id
	
	GROUP BY o.main_operation, o.sub_operation, e.energy_source
	ORDER BY o.main_operation, o.sub_operation, e.energy_source),


co2emissions_energyconsumption_energycost2 AS 
	(SELECT 
	
	   main_operation, 
	
	   sub_operation, 

	   energy_source,
	
	   CASE WHEN main_operation = 'MATERIAL MOVEMENT' THEN total_consumption_MJ/1.5
	        ELSE total_consumption_MJ
	        END AS total_consumption_MJ,
	
	   CASE WHEN main_operation = 'MATERIAL MOVEMENT' THEN total_tco2e_emissions/1.5
	        ELSE total_tco2e_emissions
	        END AS total_tco2e_emissions,

	   CASE WHEN main_operation = 'MATERIAL MOVEMENT' THEN total_energy_cost/15
	        ELSE total_energy_cost/10
	        END AS total_energy_cost
	FROM co2emissions_energyconsumption_energycost
	
	ORDER BY main_operation, sub_operation, energy_source)
	

SELECT main_operation, 
	
	   sub_operation, 

	   energy_source,
	
	   ROUND(total_consumption_MJ/(SELECT SUM(metric_value) FROM daily_production_clean WHERE metric_name = 'COPPER CONCENTRATE PRODUCED'),4)	         AS MJ_consumption_per_tonne_concentrate,
	
	   ROUND((total_tco2e_emissions*1000)/(SELECT SUM(metric_value) FROM daily_production_clean WHERE metric_name = 'COPPER CONCENTRATE PRODUCED'),4) AS kgco2e_emissions_per_tonne_concentrate,

	   ROUND(total_energy_cost/(SELECT SUM(metric_value) FROM daily_production_clean WHERE metric_name = 'COPPER CONCENTRATE PRODUCED'),5) AS energy_cost_per_tonne_concentrate
	
    FROM co2emissions_energyconsumption_energycost2
	
    ORDER BY main_operation, sub_operation, energy_source

,main_operation,sub_operation,energy_source,MJ_consumption_per_tonne_concentrate,kgco2e_emissions_per_tonne_concentrate,energy_cost_per_tonne_concentrate
0,MATERIAL MOVEMENT,Dozing and grading,DIESEL,185.3004,12.4151,0.82201
1,MATERIAL MOVEMENT,Excavating and loading,DIESEL,307.4003,20.5958,1.36447
2,MATERIAL MOVEMENT,Ore hauling,DIESEL,617.7258,41.3876,2.73144
3,MATERIAL MOVEMENT,Stockpile Reclaim,DIESEL,47.6009,3.1893,0.21128
4,MATERIAL MOVEMENT,Stockpile Reclaim,GRID ELECTRICITY,52.8241,10.1246,0.22564
5,MATERIAL MOVEMENT,Stockpile Reclaim,SOLAR PPA,8.6736,0.0723,0.02091
6,MATERIAL MOVEMENT,Waste hauling,DIESEL,924.8382,61.9642,4.10493
7,MINE DEVELOPMENT,Blast-hole drilling,DIESEL,247.3561,16.5729,1.09727
8,MINE DEVELOPMENT,Blasting support,DIESEL,55.2691,3.7030,0.24537
9,MINE DEVELOPMENT,Grade-control drilling,DIESEL,99.3839,6.6587,0.44110
